# Part 1: PySCF to MRA, plain integrals, and MRA-DMRG orbital optimization

This notebook constructs the helium calculation without importing the
repository research packages. It uses only the public PySCF, VAMPyR,
NumPy/SciPy, and block2 APIs.

The calculation has two stages:

1. Project five PySCF RHF/cc-pVDZ molecular orbitals into an adaptive MRA
   representation and verify the plain one- and two-electron integrals.
2. Optimize the real-space orbital shapes with the plain MRA-DMRG
   stationary equations, then save both the projected and optimized
   orbital families.

All quantities are in Hartree atomic units. The nucleus is the exact
point potential $V_{\mathrm{nuc}}(\mathbf r)=-2/|\mathbf r|$.


## Representation and precision

A `FunctionTree` is **not a grid-value table**. Each leaf stores local
polynomial coefficients. When a local wavelet detail is larger than the
threshold implied by `PREC`, MRCPP replaces that three-dimensional cell
by eight children. The polynomial order `ORDER` stays fixed; adaptivity
changes the local refinement level.

The tree can nevertheless be evaluated at any external Cartesian point.
Its Cartesian derivatives are obtained with the MRA derivative operator,
not by finite differences on a grid.


In [1]:
import json
import os
import pathlib
import sys
import time

here = pathlib.Path.cwd().resolve()
if (here / "tutorial").is_dir():
    TUTORIAL = here / "tutorial"
elif here.name == "tutorial":
    TUTORIAL = here
else:
    raise RuntimeError("Run this notebook from the project root or tutorial/")
os.chdir(TUTORIAL)
sys.path.insert(0, str(TUTORIAL / "tools"))

from env_provenance import write as write_provenance

provenance = write_provenance(TUTORIAL / "data" / "provenance.json")
print(json.dumps(provenance, indent=2, sort_keys=True))


{
  "policy": "commit is authoritative when present; version is the fallback when no commit is available",
  "python": "3.12.12",
  "software": {
    "block2": {
      "commit": "a7f7da9274375483ef2a6dcc28bfb50295fdd2db",
      "version": "0.5.3"
    },
    "ipykernel": {
      "version": "7.2.0"
    },
    "jupyter": {
      "version": "1.1.1"
    },
    "mrcpp": {
      "commit": "8107aabe28d6e75f04d66c95a94c157731484eae"
    },
    "nbformat": {
      "version": "5.10.4"
    },
    "numpy": {
      "version": "2.4.6"
    },
    "pyscf": {
      "version": "2.13.1"
    },
    "pytchint": {
      "commit": "4cba43b7387cd3255950e793815f34c354cee926",
      "role": "interface reference only"
    },
    "scipy": {
      "version": "1.17.1"
    },
    "vampyr": {
      "commit": "cfffb56ef83f8850cd4ee83750e41f0fa51ebf0d",
      "version": "1.0rc1"
    }
  }
}


In [2]:
import numpy as np
from scipy import linalg
from pyscf import ao2mo, fci, gto, scf
from vampyr import vampyr3d as vp

np.set_printoptions(precision=8, suppress=True)

BOX = 16.0
ORDER = int(os.environ.get("MRA_TUTORIAL_ORDER", "11"))
PREC = float(os.environ.get("MRA_TUTORIAL_PREC", "1e-5"))
BASIS = "cc-pVDZ"
N_ORBS = 5
N_ELEC = 2
NUCLEI = [(2.0, (0.0, 0.0, 0.0))]

print(
    f"He/{BASIS}: {N_ORBS} orbitals, MRA box=[-{BOX:g},{BOX:g}]^3, "
    f"k={ORDER}, precision={PREC:g}"
)


He/cc-pVDZ: 5 orbitals, MRA box=[-16,16]^3, k=11, precision=1e-05


## The plain MRA operators

The kinetic matrix is evaluated in gradient form,

$$
h_{pq}=\frac12\int\nabla\phi_p\cdot\nabla\phi_q\,d\mathbf r
+\int \phi_pV_{\mathrm{nuc}}\phi_q\,d\mathbf r .
$$

Pair densities $\rho_{pr}=\phi_p\phi_r$ are convolved with the
Poisson Green function. `PoissonOperator` applies $G_0$, so the source
is explicitly $4\pi\rho$:

$$
g_{pqrs}=\langle pq|rs\rangle
=\int \rho_{pr}(\mathbf r)\,
G_0[4\pi\rho_{qs}](\mathbf r)\,d\mathbf r .
$$

Thus `g[p,q,r,s]` is in physicists' order and
`g.transpose(0,2,1,3)` is in chemists' order.


In [3]:
class MRAWorld:
    def __init__(self, nuclei, box, order, prec):
        self.nuclei = list(nuclei)
        self.box = float(box)
        self.order = int(order)
        self.prec = float(prec)
        self.mra = vp.MultiResolutionAnalysis(
            box=[-int(box), int(box)], order=order
        )
        self.projector = vp.ScalingProjector(self.mra, prec=prec)
        self.poisson = vp.PoissonOperator(self.mra, prec=prec)
        self.derivative = vp.ABGVDerivative(self.mra, a=0.5, b=0.5)
        self._helmholtz = {}

        def point_nuclear_potential(r):
            value = 0.0
            for charge, center in self.nuclei:
                distance = np.linalg.norm(np.asarray(r) - np.asarray(center))
                value -= charge / max(float(distance), 1e-12)
            return value

        self.v_nuc = self.projector(point_nuclear_potential)
        self.e_nn = sum(
            za * zb / np.linalg.norm(np.asarray(ra) - np.asarray(rb))
            for a, (za, ra) in enumerate(self.nuclei)
            for b, (zb, rb) in enumerate(self.nuclei)
            if a < b
        )

    def project(self, function):
        return self.projector(function)

    @staticmethod
    def dot(left, right):
        return vp.dot(left, right)

    def linear_combination(self, coefficients, trees):
        terms = [
            (float(coefficient), tree)
            for coefficient, tree in zip(coefficients, trees)
            if abs(coefficient) > 1e-14
        ]
        if not terms:
            return 0.0 * trees[0]
        result = vp.FunctionTree(self.mra)
        vp.advanced.add(self.prec, result, terms)
        result.crop(self.prec)
        return result

    def lowdin(self, orbitals):
        overlap = np.array(
            [[self.dot(a, b) for b in orbitals] for a in orbitals]
        )
        eigenvalues, eigenvectors = np.linalg.eigh(overlap)
        if eigenvalues.min() < 1e-10:
            raise RuntimeError(
                f"Nearly linearly dependent orbitals: min eig(S)="
                f"{eigenvalues.min():.3e}"
            )
        inverse_half = (
            eigenvectors
            @ np.diag(eigenvalues ** -0.5)
            @ eigenvectors.T
        )
        return [
            self.linear_combination(inverse_half[:, p], orbitals)
            for p in range(len(orbitals))
        ]

    def pair_densities(self, orbitals):
        rho = {}
        for p in range(len(orbitals)):
            for q in range(p, len(orbitals)):
                value = orbitals[p] * orbitals[q]
                value.crop(self.prec)
                rho[(p, q)] = value
        return rho

    def pair_potentials(self, rho):
        potentials = {}
        for pair, density in rho.items():
            value = self.poisson(4.0 * np.pi * density)
            value.crop(self.prec)
            potentials[pair] = value
        return potentials

    def one_body(self, orbitals, rho):
        gradients = [
            vp.gradient(self.derivative, orbital) for orbital in orbitals
        ]
        matrix = np.empty((len(orbitals), len(orbitals)))
        for p in range(len(orbitals)):
            for q in range(p, len(orbitals)):
                kinetic = 0.5 * sum(
                    self.dot(gradients[p][axis], gradients[q][axis])
                    for axis in range(3)
                )
                nuclear = self.dot(rho[(p, q)], self.v_nuc)
                matrix[p, q] = matrix[q, p] = kinetic + nuclear
        return matrix, gradients

    def two_body(self, rho, potentials, norb):
        pairs = sorted(rho)
        pair_values = {}
        for p_index, p_pair in enumerate(pairs):
            for q_pair in pairs[p_index:]:
                value = self.dot(rho[p_pair], potentials[q_pair])
                pair_values[(p_pair, q_pair)] = value
                pair_values[(q_pair, p_pair)] = value

        tensor = np.empty((norb, norb, norb, norb))
        for p in range(norb):
            for q in range(norb):
                for r in range(norb):
                    for s in range(norb):
                        pr = tuple(sorted((p, r)))
                        qs = tuple(sorted((q, s)))
                        tensor[p, q, r, s] = pair_values[(pr, qs)]
        return tensor

    def integrals(self, orbitals):
        rho = self.pair_densities(orbitals)
        potentials = self.pair_potentials(rho)
        h, gradients = self.one_body(orbitals, rho)
        g = self.two_body(rho, potentials, len(orbitals))
        return h, g, rho, potentials, gradients

    def helmholtz(self, exponent):
        key = round(float(exponent), 12)
        if key not in self._helmholtz:
            self._helmholtz[key] = vp.HelmholtzOperator(
                self.mra, exp=exponent, prec=self.prec
            )
        return self._helmholtz[key]

ctx = MRAWorld(NUCLEI, BOX, ORDER, PREC)
print("MRA world and point nuclear potential constructed.")


MRA world and point nuclear potential constructed.


## PySCF reference and projection

PySCF supplies the initial RHF orbitals and an independent analytic
Gaussian-integral reference. The initialization follows the same
hierarchy as the Gaussian basis:

$$
\chi_A(\mathbf r)=\sum_a d_{Aa}g_a(\mathbf r),
\qquad
\phi_p(\mathbf r)=\sum_A C_{Ap}\chi_A(\mathbf r).
$$

`ScalingProjector` is linear, so the code projects each unique analytic
primitive once and then performs the two linear combinations in tree
form:

$$
P_{\mathrm{MRA}}\phi_p
=
\sum_A C_{Ap}
\underbrace{\sum_a d_{Aa}P_{\mathrm{MRA}}g_a}_{\text{AO tree }A}.
$$

The He/cc-pVDZ basis contains only $s$ and $p$ shells. A VAMPyR
`GaussFunc` represents an unnormalized Cartesian monomial, so the
PySCF real-spherical angular factors are inserted exactly:

$$
N_s=\frac{1}{2\sqrt{\pi}},
\qquad
N_p=\sqrt{\frac{3}{4\pi}}.
$$

No fixed grid of MO values is built. `ScalingProjector` evaluates the
analytic primitives only on its adaptive cell quadrature. The
`primitive_trees` and `ao_trees` below are temporary initialization
objects. After Löwdin orthonormalization, both the pure checkpoint and
the orbital optimizer contain only MO trees. A basis containing $d$ or
higher shells requires the exact real-spherical-to-Cartesian
transformation and is rejected by this tutorial implementation.


In [4]:
mol = gto.M(
    atom=[("He", (0.0, 0.0, 0.0))],
    basis=BASIS,
    unit="Bohr",
    charge=0,
    spin=0,
    verbose=0,
)
mean_field = scf.RHF(mol)
e_rhf = float(mean_field.kernel())
if not mean_field.converged:
    raise RuntimeError("PySCF RHF did not converge")

coefficients = np.asarray(mean_field.mo_coeff[:, :N_ORBS])
h_ao = mean_field.get_hcore()
h_reference = coefficients.T @ h_ao @ coefficients
eri_reference = ao2mo.restore(
    1, ao2mo.kernel(mol, coefficients), N_ORBS
)
g_reference = eri_reference.transpose(0, 2, 1, 3)
e_fci_reference, _ = fci.direct_spin1.kernel(
    h_reference, eri_reference, N_ORBS, N_ELEC
)

print(f"PySCF RHF energy       = {e_rhf:.10f} Ha")
print(f"PySCF FCI/cc-pVDZ      = {e_fci_reference:.10f} Ha")
print(f"Exact nonrelativistic  = {-2.903724:.10f} Ha")


PySCF RHF energy       = -2.8551604772 Ha
PySCF FCI/cc-pVDZ      = -2.8875948311 Ha
Exact nonrelativistic  = -2.9037240000 Ha


In [5]:
_POLYNOMIAL = {
    "": (0, 0, 0),
    "x": (1, 0, 0),
    "y": (0, 1, 0),
    "z": (0, 0, 1),
}

def ao_primitive_expansions(molecule):
    from pyscf.gto import gto_norm

    angular_factor = {
        0: 1.0 / (2.0 * np.sqrt(np.pi)),
        1: np.sqrt(3.0 / (4.0 * np.pi)),
    }
    labels = molecule.ao_labels(fmt=None)
    expansions = []
    ao_index = 0
    for shell in range(molecule.nbas):
        angular_momentum = molecule.bas_angular(shell)
        if angular_momentum > 1:
            raise NotImplementedError(
                "The fast tutorial projector supports only s and p shells"
            )
        exponents = molecule.bas_exp(shell)
        contractions = (
            molecule.bas_ctr_coeff(shell)
            * gto_norm(angular_momentum, exponents)[:, None]
            * angular_factor[angular_momentum]
        )
        center = tuple(float(x) for x in molecule.bas_coord(shell))
        for contraction in range(contractions.shape[1]):
            for _component in range(2 * angular_momentum + 1):
                polynomial = _POLYNOMIAL[labels[ao_index][3]]
                expansions.append(
                    [
                        (
                            float(exponent),
                            float(coefficient),
                            center,
                            polynomial,
                        )
                        for exponent, coefficient in zip(
                            exponents, contractions[:, contraction]
                        )
                    ]
                )
                ao_index += 1
    assert ao_index == molecule.nao
    return expansions

def project_molecular_orbitals(context, molecule, mo_coefficients):
    ao_expansions = ao_primitive_expansions(molecule)

    # Stage 1: project every unique analytic primitive once.
    primitive_trees = {}
    for expansion in ao_expansions:
        for exponent, _, center, polynomial in expansion:
            key = (exponent, center, polynomial)
            if key not in primitive_trees:
                gaussian = vp.GaussFunc(
                    beta=exponent,
                    alpha=1.0,
                    position=list(center),
                    poly_exponent=list(polynomial),
                )
                primitive_trees[key] = context.project(gaussian)

    # Stage 2: form each contracted spherical AO as an MRA tree.
    ao_trees = []
    for expansion in ao_expansions:
        terms = [
            (
                coefficient,
                primitive_trees[(exponent, center, polynomial)],
            )
            for exponent, coefficient, center, polynomial in expansion
        ]
        ao_tree = vp.FunctionTree(context.mra)
        vp.advanced.add(context.prec, ao_tree, terms)
        ao_tree.crop(context.prec)
        ao_trees.append(ao_tree)

    # Stage 3: form PySCF molecular orbitals by the LCAO coefficients.
    mo_trees = []
    for orbital in range(mo_coefficients.shape[1]):
        terms = [
            (
                float(mo_coefficients[ao, orbital]),
                ao_trees[ao],
            )
            for ao in range(len(ao_trees))
            if abs(mo_coefficients[ao, orbital]) > 1e-14
        ]
        tree = vp.FunctionTree(context.mra)
        vp.advanced.add(context.prec, tree, terms)
        tree.crop(context.prec)
        mo_trees.append(tree)

    # Löwdin corrects only finite-precision projection error. From this
    # point onward every active and persisted FunctionTree is an MO.
    projected_mos = context.lowdin(mo_trees)
    return projected_mos

started = time.time()
pure_orbitals = project_molecular_orbitals(ctx, mol, coefficients)
overlap_pure = np.array(
    [[ctx.dot(a, b) for b in pure_orbitals] for a in pure_orbitals]
)
print(f"Projected and orthonormalized in {time.time() - started:.1f} s")
print(
    "max|S-I| =",
    np.max(np.abs(overlap_pure - np.eye(N_ORBS))),
)


Projected and orthonormalized in 0.2 s


max|S-I| = 1.7585159863987273e-15


In [6]:
started = time.time()
h_mra, g_mra, rho_pure, potentials_pure, gradients_pure = ctx.integrals(
    pure_orbitals
)
delta_h = float(np.max(np.abs(h_mra - h_reference)))
delta_g = float(np.max(np.abs(g_mra - g_reference)))
e_fci_mra, _ = fci.direct_spin1.kernel(
    h_mra,
    g_mra.transpose(0, 2, 1, 3),
    N_ORBS,
    N_ELEC,
)
delta_energy = float(abs(e_fci_mra - e_fci_reference))

print(f"Plain MRA integrals built in {time.time() - started:.1f} s")
print(f"max|h_MRA-h_PySCF| = {delta_h:.3e}")
print(f"max|g_MRA-g_PySCF| = {delta_g:.3e}")
print(f"E_FCI[MRA h,g]      = {e_fci_mra:.10f} Ha")
print(f"|Delta E_FCI|       = {delta_energy:.3e} Ha")

# The elementwise target comes from the tutorial specification.
# At k=9, PREC=1e-5 gave max|Delta g|=2.19e-7. The executable default
# therefore raises the local polynomial order to k=11 while retaining
# the documented 1e-5 adaptive threshold.
assert delta_h <= 1e-7, delta_h
assert delta_g <= 1e-7, delta_g


Plain MRA integrals built in 5.8 s
max|h_MRA-h_PySCF| = 5.970e-11
max|g_MRA-g_PySCF| = 3.952e-08
E_FCI[MRA h,g]      = -2.8875948198 Ha
|Delta E_FCI|       = 1.129e-08 Ha


## Saving a reproducible orbital checkpoint

`saveTree` stores the adaptive coefficients, not samples on a shared
grid. `world.json` records the MRA domain, order, precision, nuclear
model, orbital order, and filenames needed by Part 2.


In [7]:
def overlap_matrix(context, orbitals):
    return np.array(
        [[context.dot(a, b) for b in orbitals] for a in orbitals]
    )

def save_orbital_family(context, orbitals, family, occupations, metadata):
    directory = TUTORIAL / "data" / f"he_ccpvdz_{family}"
    directory.mkdir(parents=True, exist_ok=True)
    records = []
    for index, orbital in enumerate(orbitals):
        basename = f"mo_{index:03d}"
        returned = pathlib.Path(
            orbital.saveTree(filename=str(directory / basename))
        )
        records.append(
            {
                "index": index,
                "basename": basename,
                "tree": returned.name,
                "occ": float(occupations[index]),
            }
        )
    world = {
        "box": context.box,
        "order": context.order,
        "prec": context.prec,
        "nuclei": [
            [float(z), [float(x) for x in center]]
            for z, center in context.nuclei
        ],
        "nuc_model": "point",
        "basis": BASIS,
        "nelec": N_ELEC,
        "orbital_family": family,
        "orbitals": records,
        "overlap": overlap_matrix(context, orbitals).tolist(),
        "provenance_file": "../provenance.json",
        **metadata,
    }
    with (directory / "world.json").open("w") as handle:
        json.dump(world, handle, indent=2, sort_keys=True)
    return directory / "world.json"

pure_world = save_orbital_family(
    ctx,
    pure_orbitals,
    "pure",
    [2, 0, 0, 0, 0],
    {
        "e_rhf": e_rhf,
        "e_fci_plain": float(e_fci_mra),
        "max_delta_h": delta_h,
        "max_delta_g": delta_g,
    },
)
print("Saved", pure_world.relative_to(TUTORIAL))


Saved data/he_ccpvdz_pure/world.json


## MRA-DMRG optimization

At fixed orbitals, block2 returns the electronic energy and spin-summed
one- and two-particle density matrices. With

$$
H=\sum h_{pq}a_p^\dagger a_q+
\frac12\sum g_{pqrs}a_p^\dagger a_q^\dagger a_sa_r,
$$

the energy derivatives are
$\partial E/\partial h=\gamma$ and
$\partial E/\partial g=\Gamma/2$. The Euler identity

$$
E_{\mathrm{elec}}=\langle h,\partial E/\partial h\rangle+
\langle g,\partial E/\partial g\rangle
$$

checks the factor and index convention on every macro-iteration.
The resulting stationary orbital equation is inverted with a bound-state
`HelmholtzOperator`, followed by Loewdin orthonormalization.


In [8]:
class Block2Solver:
    def __init__(
        self,
        nelec,
        bond_dim=250,
        num_sweeps=20,
        tolerance=1e-12,
    ):
        self.nelec = int(nelec)
        self.bond_dim = int(bond_dim)
        self.num_sweeps = int(num_sweeps)
        self.tolerance = float(tolerance)
        self.calls = 0
        self.driver = None

    def __call__(self, h, g):
        from pyblock2.driver.core import DMRGDriver, SymmetryTypes

        norb = h.shape[0]
        if self.driver is None:
            self.driver = DMRGDriver(
                scratch=str(TUTORIAL / "tmp_block2"),
                symm_type=SymmetryTypes.SU2,
                n_threads=1,
            )
        self.driver.initialize_system(
            n_sites=norb, n_elec=self.nelec, spin=0
        )
        eri_chem = np.ascontiguousarray(
            g.transpose(0, 2, 1, 3), dtype=float
        )
        mpo = self.driver.get_qc_mpo(
            h1e=np.ascontiguousarray(h, dtype=float),
            g2e=eri_chem,
            ecore=0.0,
            iprint=0,
        )
        self.calls += 1
        ket = self.driver.get_random_mps(
            tag=f"KET{self.calls}",
            bond_dim=self.bond_dim,
            nroots=1,
        )
        sweeps = self.num_sweeps
        bond_dims = [min(50, self.bond_dim)] * min(4, sweeps)
        bond_dims += [self.bond_dim] * (sweeps - len(bond_dims))
        noises = [1e-4] * min(4, sweeps)
        noises += [1e-6] * min(4, sweeps - len(noises))
        noises += [0.0] * (sweeps - len(noises))
        energy = float(
            self.driver.dmrg(
                mpo,
                ket,
                n_sweeps=sweeps,
                bond_dims=bond_dims,
                noises=noises,
                thrds=[self.tolerance] * sweeps,
                tol=self.tolerance,
                iprint=0,
            )
        )
        dh = np.asarray(self.driver.get_1pdm(ket), dtype=float)
        dg = 0.5 * np.asarray(
            self.driver.get_2pdm(ket), dtype=float
        )
        euler = float(np.sum(h * dh) + np.sum(g * dg))
        assert abs(energy - euler) <= 1e-7 * max(1.0, abs(energy)), (
            energy,
            euler,
        )
        return energy, dh, dg

def symmetrize_energy_derivatives(dh, dg):
    d1 = 0.5 * (dh + dh.T)
    d2 = 0.25 * (
        dg
        + dg.transpose(1, 0, 3, 2)
        + dg.transpose(2, 1, 0, 3)
        + dg.transpose(1, 2, 3, 0)
    )
    return d1, d2

def lagrange_multipliers(d1, d2, h, g):
    epsilon = np.einsum("mj,nj->mn", d1, h)
    epsilon += 2.0 * np.einsum("mjkl,njkl->mn", d2, g)
    return 0.5 * (epsilon + epsilon.T)


In [9]:
def update_orbitals(context, orbitals, potentials, rotation, occupations,
                    epsilon_rotated, d2):
    norb = len(orbitals)
    rotated = [
        context.linear_combination(rotation[p, :], orbitals)
        for p in range(norb)
    ]
    updated = []
    for p in range(norb):
        if abs(occupations[p]) < 1e-10:
            print(f"  [warn] occupation {p} is too small; orbital frozen")
            updated.append(rotated[p])
            continue
        ratio = epsilon_rotated[p, p] / occupations[p]
        if ratio >= 0.0:
            print(
                f"  [warn] epsilon/lambda={ratio:.4e} for orbital {p}; "
                "orbital frozen"
            )
            updated.append(rotated[p])
            continue

        exponent = np.sqrt(-2.0 * ratio)
        amplitude = np.einsum("i,ijkl->jkl", rotation[p, :], d2)
        rhs = context.v_nuc * rotated[p]
        rhs.crop(context.prec)

        for k in range(norb):
            coefficients_by_pair = {}
            for j in range(norb):
                for l in range(norb):
                    pair = tuple(sorted((j, l)))
                    coefficients_by_pair[pair] = (
                        coefficients_by_pair.get(pair, 0.0)
                        + amplitude[j, k, l]
                    )
            weighted_potential = None
            for pair, coefficient in coefficients_by_pair.items():
                if abs(coefficient) < 1e-12:
                    continue
                term = coefficient * potentials[pair]
                weighted_potential = (
                    term
                    if weighted_potential is None
                    else weighted_potential + term
                )
            if weighted_potential is not None:
                term = weighted_potential * orbitals[k]
                term.crop(context.prec)
                rhs = rhs + (2.0 / occupations[p]) * term

        for q in range(norb):
            if q == p or abs(epsilon_rotated[p, q]) < 1e-12:
                continue
            rhs = rhs - (
                epsilon_rotated[p, q] / occupations[p]
            ) * rotated[q]

        rhs.crop(context.prec)
        new_orbital = -2.0 * context.helmholtz(exponent)(rhs)
        new_orbital.crop(context.prec)
        updated.append(new_orbital)
    return context.lowdin(updated)

def optimize_orbitals(context, initial_orbitals, solver, delta=1e-5,
                      max_iter=40):
    orbitals = context.lowdin(initial_orbitals)
    history = []
    previous = None
    for iteration in range(1, max_iter + 1):
        started = time.time()
        h, g, _, potentials, _ = context.integrals(orbitals)
        electronic, dh, dg = solver(h, g)
        total = electronic + context.e_nn
        history.append(total)
        change = np.nan if previous is None else total - previous
        print(
            f"iter {iteration:2d}: E={total:.10f}  "
            f"dE={change:+.3e}  time={time.time()-started:.1f}s"
        )
        if previous is not None and abs(change) < delta:
            return total, orbitals, history
        previous = total

        d1, d2 = symmetrize_energy_derivatives(dh, dg)
        epsilon = lagrange_multipliers(d1, d2, h, g)
        occupations, vectors = np.linalg.eigh(d1)
        order = np.argsort(occupations)[::-1]
        occupations = occupations[order]
        vectors = vectors[:, order]
        rotation = vectors.T
        epsilon_rotated = rotation @ epsilon @ rotation.T
        print("  occupations:", np.array2string(occupations, precision=5))
        orbitals = update_orbitals(
            context,
            orbitals,
            potentials,
            rotation,
            occupations,
            epsilon_rotated,
            d2,
        )
    print("WARNING: maximum number of macro-iterations reached")
    return previous, orbitals, history


In [10]:
solver = Block2Solver(
    N_ELEC,
    bond_dim=int(os.environ.get("MRA_TUTORIAL_BOND_DIM", "250")),
    num_sweeps=int(os.environ.get("MRA_TUTORIAL_SWEEPS", "20")),
)
e_optimized, optimized_orbitals, optimization_history = optimize_orbitals(
    ctx,
    pure_orbitals,
    solver,
    delta=float(os.environ.get("MRA_TUTORIAL_DELTA", "1e-5")),
    max_iter=int(os.environ.get("MRA_TUTORIAL_MAX_ITER", "40")),
)
print(f"Optimized MRA-DMRG energy = {e_optimized:.10f} Ha")
print("Reference reported for He/cc-pVDZ: -2.89767133 Ha")
assert abs(e_optimized - (-2.89767133)) < 5e-4, e_optimized

h_optimized, g_optimized, _, _, _ = ctx.integrals(optimized_orbitals)
optimized_world = save_orbital_family(
    ctx,
    optimized_orbitals,
    "optimized",
    np.linalg.eigvalsh(
        solver(h_optimized, g_optimized)[1]
    )[::-1],
    {
        "e_mra_dmrg": float(e_optimized),
        "convergence_history": [float(x) for x in optimization_history],
    },
)
print("Saved", optimized_world.relative_to(TUTORIAL))


iter  1: E=-2.8875948198  dE=+nan  time=9.0s
  occupations: [1.98549 0.00832 0.00206 0.00206 0.00206]


iter  2: E=-2.8973900152  dE=-9.795e-03  time=10.1s
  occupations: [1.98431 0.00805 0.00255 0.00255 0.00255]


iter  3: E=-2.8976494910  dE=-2.595e-04  time=10.0s
  occupations: [1.98447 0.00782 0.00257 0.00257 0.00257]


iter  4: E=-2.8976690186  dE=-1.953e-05  time=10.3s
  occupations: [1.98458 0.00773 0.00257 0.00257 0.00257]


iter  5: E=-2.8976711522  dE=-2.134e-06  time=10.0s
Optimized MRA-DMRG energy = -2.8976711522 Ha
Reference reported for He/cc-pVDZ: -2.89767133 Ha


Saved data/he_ccpvdz_optimized/world.json


## Part 1 checkpoint

Part 2 needs only `world.json` and the `.tree` files. The initial
Gaussian coefficients are not needed after projection. The optimized
trees are also not assumed to remain in the original Gaussian span.
